# 19_Gradient Boosting

In [1]:
import pandas as pd
from sklearn.datasets import load_iris, load_wine, load_breast_cancer
from sklearn.datasets import fetch_california_housing
from google.colab import drive
import os

# Google Drive Mount
drive.mount('/content/drive')

# Dataset을 저장할 Folder
data_path = "/content/drive/My Drive/Colab Notebooks/0_ML/Input/"

Mounted at /content/drive


## Gradient Boosting

AdaBoost는 이전 Weak Learner가 잘못 Prediction한 Samples의 Weight를 증가시키고 다음 Weak Learner를 Training하였다.

Gradient Boosting도 여러 Weak Learners를 순차적으로 Training한다.

그러나 다음 Weak Learner를 만드는 방법이 다르다.

Gradient Boosting의 기본 아이디어는 다음과 같다.

> 현재 Model의 Error를 다음 Weak Learner가 보완하도록 순차적으로 Model을 추가한다.

## From AdaBoost to Gradient Boosting

AdaBoost:

> Wrong Samples → Increase Sample Weights → Next Weak Learner

Gradient Boosting:

> Current Prediction → Error → Learn the Error → Add New Weak Learner

두 방법 모두 Weak Learners를 순차적으로 추가하지만, 이전 Model의 Error를 처리하는 방법이 다르다.

## Learning the Residual

실제 값이 다음과 같다고 하자.

$y = 10$

현재 Model의 Prediction이

$\hat{y} = 7$

이라면 Residual은

$$
r = y-\hat{y}=3
$$

이다.

다음 Weak Learner는 이 Residual을 Prediction하도록 Training할 수 있다.

만약 다음 Weak Learner가

$\hat{r}=2.5$

를 Prediction했다면 새로운 Prediction은

$$
\hat{y}_{new}=7+2.5=9.5
$$

가 된다.

즉, 새로운 Weak Learner가 이전 Model의 부족한 부분을 보완한다.

## Sequential Learning

Gradient Boosting에서는 하나의 Model을 만든 후 끝내지 않는다.

현재 Model의 Error를 이용하여 새로운 Weak Learner를 순차적으로 추가한다.

> Model 1 → Error → Model 2 → Remaining Error → Model 3 → ...

각 Weak Learner는 이전까지 만들어진 Model이 충분히 설명하지 못한 부분을 학습한다.

최종 Prediction은 여러 Weak Learners의 결과를 순차적으로 결합하여 만들어진다.

## How Does Gradient Boosting Learn?

다음과 같은 간단한 Regression Problem을 생각해 보자.

| X | Actual y |
|---:|---:|
| 1 | 10 |
| 2 | 12 |
| 3 | 18 |
| 4 | 20 |
| 5 | 25 |

Gradient Boosting은 하나의 복잡한 Model을 한 번에 만드는 대신,
현재 Model의 Prediction Error를 새로운 Weak Learner가 계속 보완한다.

In [11]:
import numpy as np

X = np.array([1, 2, 3, 4, 5])
y = np.array([10, 12, 18, 20, 25])

prediction = np.mean(y)

print("Initial Prediction:", prediction)

Initial Prediction: 17.0


## Calculate the Residual

현재 Model의 Prediction이 17이라면 Residual은

$$
r_i = y_i-\hat{y}_i
$$

이다.

In [12]:
residual = y - prediction

for x, actual, r in zip(X, y, residual):
    print(
        f"X={x}, "
        f"Actual={actual}, "
        f"Prediction={prediction:.1f}, "
        f"Residual={r:.1f}"
    )

X=1, Actual=10, Prediction=17.0, Residual=-7.0
X=2, Actual=12, Prediction=17.0, Residual=-5.0
X=3, Actual=18, Prediction=17.0, Residual=1.0
X=4, Actual=20, Prediction=17.0, Residual=3.0
X=5, Actual=25, Prediction=17.0, Residual=8.0


### 설명

> 첫 Model은 모두 17이라고 Prediction했습니다.
그런데 X=1에서는 7만큼 너무 크게 Prediction했고, X=5에서는 8만큼 너무 작게 Prediction했다.
그렇다면 두 번째 Model에게 무엇을 학습시키면 될까?
y를 처음부터 다시 학습시키는 것이 아니라 바로 이 부족한 부분을 학습시키는 것이다.

In [9]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(
    max_depth=1,
    random_state=42
)

tree.fit(
    X.reshape(-1, 1),           # 컬럼벡터형태로 입력형태를 맞추어야 해서
    residual
)
# 다음 문제를 학습하는 것
# X  →  Residual

# 1  →  -7
# 2  →  -5
# 3  →   1
# 4  →   3
# 5  →   8

# max_depth=1이므로 Tree는 딱 한 번만 Split할 수 있다.
# 이 데이터에서는 대략 다음과 같은 Rule을 찾게 된다.
#          X <= 2.5 ?
#          /        \
#        Yes         No
#       /             \
#   -6.0              4.0

correction = tree.predict(
    X.reshape(-1, 1)
)

print("Correction:", correction)

Correction: [-6. -6.  4.  4.  4.]


## Learning Rate

새로운 Weak Learner의 Prediction을 그대로 모두 반영할 필요는 없다.

Gradient Boosting에서는 Learning Rate를 이용하여 새로운 Weak Learner의 영향을 조절한다.

$$
F_{new}(X)
=
F_{old}(X)
+
\eta h(X)
$$

여기서

- $F_{old}(X)$: 현재 Model의 Prediction
- $h(X)$: 새 Weak Learner의 Prediction
- $\eta$: Learning Rate

Learning Rate가 작으면 새로운 Weak Learner의 결과를 조금씩 반영한다.

Learning Rate가 크면 새로운 Weak Learner의 결과를 더 크게 반영한다.

In [10]:
learning_rate = 0.5

new_prediction = (
    prediction
    + learning_rate * correction
)

new_residual = (
    y - new_prediction
)

for x, actual, pred, r in zip(
    X,
    y,
    new_prediction,
    new_residual
):
    print(
        f"X={x}, "
        f"Actual={actual}, "
        f"New Prediction={pred:.2f}, "
        f"New Residual={r:.2f}"
    )

X=1, Actual=10, New Prediction=14.00, New Residual=-4.00
X=2, Actual=12, New Prediction=14.00, New Residual=-2.00
X=3, Actual=18, New Prediction=19.00, New Residual=-1.00
X=4, Actual=20, New Prediction=19.00, New Residual=1.00
X=5, Actual=25, New Prediction=19.00, New Residual=6.00


## Add Another Weak Learner

첫 번째 Weak Learner가 모든 Error를 제거한 것은 아니다.

새로운 Prediction에도 Residual이 남아 있다.

따라서 다시 남아 있는 Residual을 이용하여 새로운 Weak Learner를 Training한다.

> Prediction → Residual → Weak Learner → Updated Prediction → New Residual → ...

## Gradient Boosting for Classification

Classification에서도 기본 아이디어는 동일하다.

현재 Model의 Prediction Error를 줄이도록 새로운 Weak Learner를 순차적으로 추가한다.

실제 Gradient Boosting Classification에서는 단순한 $y-\hat{y}$ 대신 Loss Function을 이용하여 다음 Weak Learner가 학습할 값을 결정한다.

이번 강의에서는 다음 핵심 아이디어에 집중한다.

> Each new Weak Learner learns how to reduce the Error of the current Model.

## Gradient Boosting Classification

Classification Data에 Gradient Boosting을 적용한다.

먼저 Data를 Training Data와 Test Data로 나눈다.

In [14]:
import pandas as pd

from sklearn.model_selection import train_test_split

data = pd.read_csv(
    data_path+"gradient_data.csv"
)

X = data.drop(
    columns=["y"]
)

y = data["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Data:", X_train.shape)
print("Test Data:", X_test.shape)

Training Data: (960, 6)
Test Data: (240, 6)


In [15]:
for lr in [0.01, 0.1, 0.5]:

    for n in [10, 50, 100]:

        model = GradientBoostingClassifier(
            n_estimators=n,
            learning_rate=lr,
            random_state=42
        )

        model.fit(X_train, y_train)

        print(
            f"Learning Rate={lr}, "
            f"Estimators={n}, "
            f"Accuracy={model.score(X_test, y_test):.3f}"
        )

Learning Rate=0.01, Estimators=10, Accuracy=0.571
Learning Rate=0.01, Estimators=50, Accuracy=0.617
Learning Rate=0.01, Estimators=100, Accuracy=0.625
Learning Rate=0.1, Estimators=10, Accuracy=0.633
Learning Rate=0.1, Estimators=50, Accuracy=0.642
Learning Rate=0.1, Estimators=100, Accuracy=0.654
Learning Rate=0.5, Estimators=10, Accuracy=0.662
Learning Rate=0.5, Estimators=50, Accuracy=0.821
Learning Rate=0.5, Estimators=100, Accuracy=0.817


## Train Gradient Boosting

`GradientBoostingClassifier`를 이용하여 Model을 Training한다.

먼저 기본적인 Learning Rate를 사용한다.

In [16]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

y_pred = model.predict(
    X_test
)

accuracy = accuracy_score(
    y_test,
    y_pred
)

print(
    "Gradient Boosting Accuracy:",
    f"{accuracy:.3f}"
)

Gradient Boosting Accuracy: 0.654


## Random Forest vs Gradient Boosting

Random Forest와 Gradient Boosting은 모두 여러 Decision Trees를 이용할 수 있다.

그러나 Trees를 Training하는 방법이 다르다.

Random Forest:

> Trees를 서로 비교적 독립적으로 Training하고 Prediction을 결합한다.

Gradient Boosting:

> Trees를 순차적으로 Training하며 새로운 Tree가 이전 Model의 Error를 보완한다.

In [4]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(
    X_train,
    y_train
)

rf_accuracy = rf.score(
    X_test,
    y_test
)

print(
    "Random Forest:",
    f"{rf_accuracy:.3f}"
)

print(
    "Gradient Boosting:",
    f"{accuracy:.3f}"
)

Random Forest: 0.783
Gradient Boosting: 0.654


## Change the Learning Rate

Learning Rate는 각각의 새로운 Weak Learner가 기존 Model을 얼마나 크게 수정할 것인지를 결정한다.

여러 Learning Rates를 사용하여 Test Accuracy의 변화를 확인해 보자.

In [5]:
learning_rates = [
    0.01,
    0.05,
    0.1,
    0.2,
    0.5
]

for lr in learning_rates:

    model = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=lr,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    accuracy = model.score(
        X_test,
        y_test
    )

    print(
        f"Learning Rate={lr}, "
        f"Accuracy={accuracy:.3f}"
    )

Learning Rate=0.01, Accuracy=0.625
Learning Rate=0.05, Accuracy=0.642
Learning Rate=0.1, Accuracy=0.654
Learning Rate=0.2, Accuracy=0.762
Learning Rate=0.5, Accuracy=0.817


## AdaBoost vs Gradient Boosting

두 Methods 모두 Weak Learners를 순차적으로 Training한다.

AdaBoost:

> 이전 Weak Learner가 틀린 Samples의 Weight를 증가시킨다.

Gradient Boosting:

> 현재 Model의 Error를 줄이는 방향으로 새로운 Weak Learner를 추가한다.

따라서 두 방법의 공통점은

> Sequential Learning

이고, 차이점은

> How the next Weak Learner corrects the previous Errors

이다.

In [6]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = data.drop(columns=["y"])
y = data["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# AdaBoost
ada = AdaBoostClassifier(
    n_estimators=100,
    random_state=42
)

ada.fit(X_train, y_train)

ada_accuracy = accuracy_score(
    y_test,
    ada.predict(X_test)
)

# Gradient Boosting
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

gb.fit(X_train, y_train)

gb_accuracy = accuracy_score(
    y_test,
    gb.predict(X_test)
)

print("AdaBoost Accuracy:", ada_accuracy)
print("Gradient Boosting Accuracy:", gb_accuracy)

AdaBoost Accuracy: 0.6125
Gradient Boosting Accuracy: 0.7375


## Summary

Gradient Boosting은 여러 Weak Learners를 순차적으로 결합한다.

기본 아이디어는 다음과 같다.

> Prediction → Error → Learn the Error → Add Weak Learner → Repeat

핵심 사항:

- Weak Learners를 순차적으로 Training한다.
- 새로운 Weak Learner는 현재 Model의 Error를 줄이도록 학습한다.
- 여러 Weak Learners의 결과를 누적하여 최종 Prediction을 만든다.
- `learning_rate`는 새로운 Weak Learner의 영향을 조절한다.
- Random Forest와 달리 각 Tree가 이전 Model의 결과에 의존한다.

## Homework - Problem 19

`Input/gradient_data2.csv`를 이용하여 Gradient Boosting을 수행한다.

다음 조건을 사용한다.

- `test_size=0.2`
- `random_state=42`
- `n_estimators=100`
- `learning_rate = 0.01, 0.05, 0.1, 0.2`

다음 질문에 답하시오.

- 각 `learning_rate`의 Test Accuracy를 소수점 셋째 자리까지 적으시오.
- 가장 높은 Test Accuracy를 보인 `learning_rate`를 모두 적으시오.

답만 제출하시오.